# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjdinSalihovic/FlyRank-ML/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
LANE = "Lane 2: Refresh / Content Opportunity Scoring"
print(LANE)

Lane 2: Refresh / Content Opportunity Scoring


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [6]:

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
decision = "which page to review first for refresh"
action = "editor opens ranked queue, refreshes/expands/protects/monitors top pages"
cost_of_wrong_call = "false positive = wasted editor hours; false negative = continued click loss"
print(decision, "->", action, "| cost:", cost_of_wrong_call)

which page to review first for refresh -> editor opens ranked queue, refreshes/expands/protects/monitors top pages | cost: false positive = wasted editor hours; false negative = continued click loss


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Starter dataset shape:", df.shape)
print("Clients:", df["client_id"].nunique())

# Gotcha from the flyrank-data skill: avg_position == 0 means "no data", not rank zero.
# A "visible" page is one FlyRank actually has position data for, ranking in the top 20.
visible = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20)]
pct_visible = 100 * len(visible) / len(df)
print(f"\nNumber 1 - Visible pages: {len(visible):,} of {len(df):,} rows ({pct_visible:.1f}%) "
      f"have real, ranked search visibility (position 1-20).")

# trend_direction is a same-window proxy (computed from trend_pct), NOT a future outcome -
# I'm using it here only to show why a single-column rule is too blunt, not as a target.
declining_visible = visible[visible["trend_direction"] == "down"]
pct_declining = 100 * len(declining_visible) / len(visible)
print(f"\nNumber 2 - Of those visible pages, {len(declining_visible):,} ({pct_declining:.1f}%) "
      f"currently show trend_direction == 'down'. That's too large a share to hand-review as a "
      f"weekly queue - a single-signal rule does not separate the pages worth an editor's time.")

# Combine staleness + decline + real demand: a small, more defensible starting queue.
stale_declining_with_demand = visible[
    (visible["freshness_tier"].isin(["91-180", "181+"])) &
    (visible["trend_direction"] == "down") &
    (visible["impressions_90d"] >= 100)
]
pct_of_visible = 100 * len(stale_declining_with_demand) / len(visible)
print(f"\nNumber 3 - Visible pages that are ALSO stale (not touched in 91+ days), declining, "
      f"and have real demand (>=100 impressions/90d): {len(stale_declining_with_demand):,} "
      f"({pct_of_visible:.1f}% of visible pages). That's a much more plausible weekly review "
      f"queue size than {len(declining_visible):,}.")


Starter dataset shape: (30000, 44)
Clients: 32

Number 1 - Visible pages: 20,256 of 30,000 rows (67.5%) have real, ranked search visibility (position 1-20).

Number 2 - Of those visible pages, 11,744 (58.0%) currently show trend_direction == 'down'. That's too large a share to hand-review as a weekly queue - a single-signal rule does not separate the pages worth an editor's time.

Number 3 - Visible pages that are ALSO stale (not touched in 91+ days), declining, and have real demand (>=100 impressions/90d): 3,366 (16.6% of visible pages). That's a much more plausible weekly review queue size than 11,744.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick public-safety sanity check: confirm no raw/identifying columns are present
# in the starter dataset before I go any further with it.
unsafe_markers = ["url", "domain", "title", "query", "name", "email"]
flagged_columns = [c for c in df.columns if any(m in c.lower() for m in unsafe_markers)]
print("Columns matching unsafe-content markers:", flagged_columns)
assert flagged_columns == [], "Found a column that looks unsafe to display publicly."
print("No raw URL/title/query/name-like columns in the starter dataset - safe to keep exploring.")


Columns matching unsafe-content markers: []
No raw URL/title/query/name-like columns in the starter dataset - safe to keep exploring.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.